# 05. External test on CCMT (Ghana), read once
The ten frozen checkpoints from notebook 04b are applied once to the raw CCMT cassava photographs (Mensah et al., 2023), with no retraining and no refitting.
1. Class mapping (Table 5): bacterial blight to CBB, green mite to CGM, mosaic to CMD, healthy to healthy. Brown spot has no counterpart and is excluded. The model keeps all five outputs, so a CBSD prediction on a CCMT image counts as an error.
2. Near-duplicate check against Makerere with the same perceptual hash as notebook 01; exact matches are removed before any prediction is made.
3. Each checkpoint uses its own temperature and threshold, fitted on the validation fit half in notebook 04b. Nothing is fitted on CCMT.
4. Macro-F1 is averaged over the four shared classes, and reported again without CGM because the competition label file calls that class green mottle.

Before running: upload `ccmt_cassava_raw.zip` (the five raw cassava folders, 7,508 images, about 370 MB) to `MyDrive/cassacheck_data/`. If Drive is full, upload the same zip through the Colab Files panel instead (it is lost when the session restarts). No Kaggle download is needed.

In [ ]:
# Cell 1: environment and Drive (checkpoints and frozen values come from notebooks 03 and 04b)
from google.colab import drive
drive.mount("/content/drive")
!pip install --quiet imagehash scikit-learn
import os, json, glob, torch, numpy as np, pandas as pd, torch.nn as nn
import matplotlib.pyplot as plt
RUNS_DIRECTORY = "/content/drive/MyDrive/cassacheck_runs"
RESULTS_DIRECTORY = "/content/drive/MyDrive/cassacheck_results"
CCMT_ZIP = "/content/drive/MyDrive/cassacheck_data/ccmt_cassava_raw.zip"
assert os.path.exists(f"{RESULTS_DIRECTORY}/frozen_models.json"), "run notebook 04b first: nothing is frozen yet"
if not os.path.exists(CCMT_ZIP): CCMT_ZIP = "/content/ccmt_cassava_raw.zip"   # fallback: the same zip uploaded through this session's Files panel
assert os.path.exists(CCMT_ZIP), "upload ccmt_cassava_raw.zip to MyDrive/cassacheck_data/, or to this Colab session with the Files panel"
print("CCMT zip:", CCMT_ZIP)
!unzip -q -o "{CCMT_ZIP}" -d /content
device = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ["CBB", "CBSD", "CGM", "CMD", "healthy"]
print("device:", device)

In [ ]:
# Cell 2: the CCMT cassava images and the class mapping (Table 5); brown spot has no counterpart and is excluded
CCMT_DIRECTORY = "/content/ccmt_cassava_raw"
CCMT_TO_LABEL = {"bacterial blight": 0, "green mite": 2, "mosaic": 3, "healthy": 4}   # nothing in CCMT maps to CBSD (1)
all_counts = {folder: len(os.listdir(f"{CCMT_DIRECTORY}/{folder}")) for folder in sorted(os.listdir(CCMT_DIRECTORY))}
print("raw CCMT cassava images per folder:", all_counts, "| total", sum(all_counts.values()))
external_frame = pd.DataFrame([{"path": f"{CCMT_DIRECTORY}/{folder}/{name}", "ccmt_class": folder, "label": label}
                               for folder, label in CCMT_TO_LABEL.items() for name in sorted(os.listdir(f"{CCMT_DIRECTORY}/{folder}"))])
print("used, brown spot excluded:", len(external_frame))
print(external_frame.ccmt_class.value_counts())

In [ ]:
# Cell 3: near-duplicate check against Makerere (perceptual hash, hash size 8, as in notebook 01). Uses no labels and no predictions.
import imagehash
from PIL import Image
from tqdm import tqdm
!wget --quiet --output-document /content/split_v1_seed42.csv https://raw.githubusercontent.com/Justine-abc/cassavacheck/main/data/split_v1_seed42.csv
makerere = pd.read_csv("/content/split_v1_seed42.csv")
external_frame["phash"] = [str(imagehash.phash(Image.open(path).convert("RGB"), hash_size=8)) for path in tqdm(external_frame.path)]
exact_match = external_frame.phash.isin(set(makerere.phash))
print("CCMT images with exactly the fingerprint of a Makerere image (removed):", int(exact_match.sum()))

# for information only: the smallest number of differing bits between each CCMT image and any Makerere image
def to_bits(hashes): return np.unpackbits(np.stack([np.frombuffer(bytes.fromhex(h), dtype=np.uint8) for h in hashes]), axis=1).astype(bool)
makerere_bits, external_bits = to_bits(makerere.phash), to_bits(external_frame.phash)
nearest = np.concatenate([(external_bits[i:i + 256, None, :] != makerere_bits[None]).sum(2).min(1) for i in range(0, len(external_bits), 256)])
print("CCMT images within 2 / 4 / 8 bits of a Makerere image:", int((nearest <= 2).sum()), "/", int((nearest <= 4).sum()), "/", int((nearest <= 8).sum()))
print("CCMT images sharing a fingerprint with another CCMT image:", int(external_frame.phash.duplicated(keep=False).sum()))

external_frame["nearest_makerere_bits"] = nearest
external_frame.to_csv(f"{RESULTS_DIRECTORY}/ccmt_manifest.csv", index=False)
external_frame = external_frame[~exact_match].reset_index(drop=True)
print("external set after the duplicate check:", len(external_frame))

In [ ]:
# Cell 4: metric functions, identical to notebook 04b (ECE with 15 bins)
from sklearn.metrics import f1_score, classification_report, confusion_matrix

def expected_calibration_error(probabilities, targets, bins=15):
    confidence, prediction = probabilities.max(1)
    correct = (prediction == targets).float()
    edges = torch.linspace(0, 1, bins + 1); error = 0.0
    for low, high in zip(edges[:-1], edges[1:]):
        in_bin = (confidence > low) & (confidence <= high)
        if in_bin.any(): error += in_bin.float().mean() * (correct[in_bin].mean() - confidence[in_bin].mean()).abs()
    return float(error)

def selective_metrics(probabilities, targets, threshold):
    confidence, prediction = probabilities.max(1)
    kept = confidence >= threshold
    coverage = float(kept.float().mean())
    selective_accuracy = float((prediction[kept] == targets[kept]).float().mean()) if kept.any() else float("nan")
    abstained_accuracy = float((prediction[~kept] == targets[~kept]).float().mean()) if (~kept).any() else float("nan")
    return coverage, selective_accuracy, abstained_accuracy

SHARED = [0, 2, 3, 4]        # CBB, CGM, CMD, healthy
WITHOUT_CGM = [0, 3, 4]      # CBB, CMD, healthy

In [ ]:
# Cell 5: frozen values from notebook 04b, the model code, and the images decoded once (same preprocessing as training and the internal test)
from torchvision import transforms, models
frozen = json.load(open(f"{RESULTS_DIRECTORY}/frozen_models.json"))
calibration_table = pd.read_csv(f"{RESULTS_DIRECTORY}/calibration_abstention_validation_halves.csv")
print(json.dumps(frozen, indent=1))
print(calibration_table[["backbone", "seed", "temperature", "threshold"]].to_string(index=False))

eval_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def build_model(backbone):
    if backbone == "mobilenetv3_large":
        network = models.mobilenet_v3_large(weights=None); network.classifier[3] = nn.Linear(network.classifier[3].in_features, 5)
    else:
        network = models.resnet18(weights=None); network.fc = nn.Linear(network.fc.in_features, 5)
    return network.to(device).eval()

# every checkpoint sees exactly the same tensors; float16 keeps the 6,000 images inside Colab memory
images = torch.stack([eval_transform(Image.open(path).convert("RGB")).half() for path in tqdm(external_frame.path)])
targets = torch.tensor(external_frame.label.values)

def external_logits(backbone, checkpoint):
    network = build_model(backbone); network.load_state_dict(torch.load(f"{RUNS_DIRECTORY}/{checkpoint}", map_location=device))
    with torch.no_grad():
        return torch.cat([network(images[i:i + 256].to(device).float()).cpu() for i in range(0, len(images), 256)])

Nothing above this line changes after cell 6 has been run. If cell 6 is run again it loads the saved logits instead of predicting again.

In [ ]:
# Cell 6: THE EXTERNAL SET, READ ONCE. All ten checkpoints, each with its own frozen temperature and threshold.
def external_metrics(logits, temperature, threshold):
    before = torch.softmax(logits, 1); after = torch.softmax(logits / temperature, 1); prediction = after.argmax(1)
    targets_np, prediction_np, no_cgm = targets.numpy(), prediction.numpy(), (targets != 2).numpy()
    coverage, selective_accuracy, abstained_accuracy = selective_metrics(after, targets, threshold)
    return {"accuracy": float((prediction == targets).float().mean()),
            "macro_f1_shared4": float(f1_score(targets_np, prediction_np, labels=SHARED, average="macro", zero_division=0)),
            "macro_f1_without_cgm": float(f1_score(targets_np[no_cgm], prediction_np[no_cgm], labels=WITHOUT_CGM, average="macro", zero_division=0)),
            "cbsd_prediction_rate": float((prediction == 1).float().mean()),
            "ece_before": expected_calibration_error(before, targets), "ece_after": expected_calibration_error(after, targets),
            "coverage": coverage, "selective_accuracy": selective_accuracy, "abstained_accuracy": abstained_accuracy}

external_rows, external_logits_store = [], {}
for _, run in calibration_table.iterrows():
    name = f"{run.backbone}_seed{run.seed}"
    saved_path = f"{RESULTS_DIRECTORY}/ccmt_logits_{name}.pt"
    if os.path.exists(saved_path):
        logits = torch.load(saved_path)
    else:
        logits = external_logits(run.backbone, f"{name}_best.pt"); torch.save(logits, saved_path)
    external_logits_store[(run.backbone, run.seed)] = logits
    metrics = external_metrics(logits, run.temperature, run.threshold)
    external_rows.append({"backbone": run.backbone, "seed": run.seed, **{key: round(value, 4) for key, value in metrics.items()}})
external_table = pd.DataFrame(external_rows)
print(external_table.to_string(index=False)); print()
print(external_table.groupby("backbone")[["accuracy", "macro_f1_shared4", "macro_f1_without_cgm", "cbsd_prediction_rate", "ece_before", "ece_after", "coverage", "selective_accuracy"]].agg(["mean", "std"]).round(4))
external_table.to_csv(f"{RESULTS_DIRECTORY}/external_ccmt_all_runs.csv", index=False)

In [ ]:
# Cell 7: the two frozen models in detail: per-class F1 against the internal test, confusion matrix (4 true classes by 5 outputs), bootstrap 95% intervals over CCMT images
rng = np.random.default_rng(42)
def bootstrap_interval(statistic, n_resamples=1000):
    values = [statistic(rng.integers(0, len(targets), len(targets))) for _ in range(n_resamples)]
    return round(float(np.nanpercentile(values, 2.5)), 4), round(float(np.nanpercentile(values, 97.5)), 4)

# per-class F1 on the internal test (3,210 images), copied from the printed output of notebook 04b cell 8
INTERNAL_PER_CLASS_F1 = {"mobilenetv3_large": {"CBB": 0.5436, "CGM": 0.7597, "CMD": 0.9346, "healthy": 0.6559},
                         "resnet18": {"CBB": 0.5326, "CGM": 0.7524, "CMD": 0.9377, "healthy": 0.6658}}
SHARED_NAMES = [CLASS_NAMES[label] for label in SHARED]

detail = {}
for backbone, spec in frozen.items():
    logits = external_logits_store[(backbone, spec["seed"])]
    before = torch.softmax(logits, 1); after = torch.softmax(logits / spec["temperature"], 1)
    prediction = after.argmax(1); confidence = after.max(1).values; kept = confidence >= spec["threshold"]
    correct = (prediction == targets).float().numpy(); targets_np = targets.numpy(); prediction_np = prediction.numpy(); kept_np = kept.numpy()
    macro_f1 = f1_score(targets_np, prediction_np, labels=SHARED, average="macro", zero_division=0)
    accuracy_ci = bootstrap_interval(lambda idx: correct[idx].mean())
    macro_f1_ci = bootstrap_interval(lambda idx: f1_score(targets_np[idx], prediction_np[idx], labels=SHARED, average="macro", zero_division=0))
    ece_after_ci = bootstrap_interval(lambda idx: expected_calibration_error(after[idx], targets[idx]))
    selective_ci = bootstrap_interval(lambda idx: correct[idx][kept_np[idx]].mean() if kept_np[idx].any() else np.nan)
    print(f"=== {backbone}, seed {spec['seed']}, T={spec['temperature']:.3f}, threshold={spec['threshold']:.3f} (CCMT, {len(targets):,} images) ===")
    print(f"accuracy {correct.mean():.4f} CI {accuracy_ci} | macro-F1 over the four shared classes {macro_f1:.4f} CI {macro_f1_ci}")
    print(f"ECE before {expected_calibration_error(before, targets):.4f} | after {expected_calibration_error(after, targets):.4f} CI {ece_after_ci}")
    print(f"coverage {kept.float().mean():.4f} | selective accuracy {correct[kept_np].mean():.4f} CI {selective_ci} | abstained accuracy {correct[~kept_np].mean():.4f}")
    print(f"share of CCMT images predicted as CBSD (always an error): {(prediction_np == 1).mean():.4f}")

    comparison = pd.DataFrame({"internal_f1": [INTERNAL_PER_CLASS_F1[backbone][name] for name in SHARED_NAMES],
                               "external_f1": f1_score(targets_np, prediction_np, labels=SHARED, average=None, zero_division=0).round(4),
                               "external_images": [int((targets_np == label).sum()) for label in SHARED]}, index=SHARED_NAMES)
    comparison["drop"] = (comparison.internal_f1 - comparison.external_f1).round(4)
    print(comparison.to_string())
    print(f"macro-F1 over the same four classes: internal {comparison.internal_f1.mean():.4f} -> external {comparison.external_f1.mean():.4f}")
    print(classification_report(targets_np, prediction_np, labels=SHARED, target_names=SHARED_NAMES, digits=4, zero_division=0))

    matrix = confusion_matrix(targets_np, prediction_np, labels=list(range(5)))[SHARED]
    fig, axis = plt.subplots(figsize=(6, 4.5)); axis.imshow(matrix, cmap="Blues")
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            axis.text(column, row, matrix[row, column], ha="center", va="center", color="white" if matrix[row, column] > matrix.max() / 2 else "black")
    axis.set_xticks(range(5), CLASS_NAMES, rotation=45); axis.set_yticks(range(4), SHARED_NAMES); axis.set_xlabel("predicted (five outputs)"); axis.set_ylabel("CCMT class")
    axis.set_title(f"CCMT confusion matrix, {backbone} seed {spec['seed']}"); plt.tight_layout(); plt.savefig(f"{RESULTS_DIRECTORY}/ccmt_confusion_{backbone}.png", dpi=150); plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for axis, (title, probabilities) in zip(axes[:2], (("Uncalibrated", before), ("Calibrated", after))):
        conf, pred = probabilities.max(1); corr = (pred == targets).float(); edges = np.linspace(0, 1, 11); centres, accs = [], []
        for low, high in zip(edges[:-1], edges[1:]):
            in_bin = (conf > low) & (conf <= high)
            if in_bin.any(): centres.append(float(conf[in_bin].mean())); accs.append(float(corr[in_bin].mean()))
        axis.plot([0, 1], [0, 1], "--", color="grey"); axis.bar(centres, accs, width=0.08, color="#2F4F6F"); axis.set_title(f"{title} (CCMT)"); axis.set_xlabel("confidence"); axis.set_ylabel("accuracy")
    order = torch.argsort(confidence, descending=True); sorted_correct = correct[order.numpy()]
    cov = np.arange(1, len(order) + 1) / len(order); risk = 1 - np.cumsum(sorted_correct) / np.arange(1, len(order) + 1)
    axes[2].plot(cov, risk, color="#2F4F6F"); axes[2].axvline(float(kept.float().mean()), color="#BFA14A", linestyle="--"); axes[2].set_xlabel("coverage"); axes[2].set_ylabel("error on retained"); axes[2].set_title("Risk-coverage (CCMT)")
    plt.tight_layout(); plt.savefig(f"{RESULTS_DIRECTORY}/ccmt_calibration_{backbone}.png", dpi=150); plt.show()

    detail[backbone] = {"images": int(len(targets)), "accuracy": float(correct.mean()), "accuracy_ci": accuracy_ci,
                        "macro_f1_shared4": float(macro_f1), "macro_f1_shared4_ci": macro_f1_ci,
                        "internal_macro_f1_shared4": float(comparison.internal_f1.mean()), "per_class": comparison.to_dict(orient="index"),
                        "cbsd_prediction_rate": float((prediction_np == 1).mean()),
                        "ece_before": expected_calibration_error(before, targets), "ece_after": expected_calibration_error(after, targets), "ece_after_ci": ece_after_ci,
                        "coverage": float(kept.float().mean()), "selective_accuracy": float(correct[kept_np].mean()), "selective_accuracy_ci": selective_ci,
                        "abstained_accuracy": float(correct[~kept_np].mean())}
json.dump(detail, open(f"{RESULTS_DIRECTORY}/external_ccmt_frozen_models.json", "w"), indent=2)
print("download the CSVs, JSONs and PNGs whose names start with ccmt_ or external_ from", RESULTS_DIRECTORY)

Report mapping: cell 3 is the cross-dataset duplicate check (3.2.1); cell 6 is E2 for all ten runs (macro-F1 over the four shared classes, with and without CGM, seed spread) and the external half of E3 and E4 (ECE before and after, realized coverage, selective accuracy); cell 7 is the frozen models in detail with bootstrap intervals over CCMT images and the per-class drop from the internal test (RQ3). Every number here is reported once, whether or not it meets a target in Table 6.